<a href="https://colab.research.google.com/github/nicowilliamxvii/TQHDC_CS441/blob/main/house_price_data_tloc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import pandas as pd

# Đọc file CSV
df = pd.read_csv('/content/House Price Prediction Dataset.csv')

# Kiểm tra 5 dòng đầu
print(df.head())

# Kiểm tra kiểu dữ liệu từng cột
print(df.dtypes)

# Kiểm tra riêng Location và Condition
print("Location data type:", df['Location'].dtype)
print("Condition data type:", df['Condition'].dtype)

# Xem các giá trị unique của Location
print("Unique Locations:", df['Location'].unique())

# Xem các giá trị unique của Condition
print("Unique Conditions:", df['Condition'].unique())

# Kiểm tra Garage (cũng là categorical)
print("Unique Garage values:", df['Garage'].unique())

# **Observation **

"Giá nhà được quyết định bởi yếu tố nào — diện tích vật lý, vị trí địa lý, hay tình trạng ngôi nhà?"

Q1 — Giá nhà phân bổ như thế nào?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

df = pd.read_csv('/content/House Price Prediction Dataset.csv')

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(df['Price'], bins=10, color='#378ADD', edgecolor='white', linewidth=0.8)

ax.set_title('Phân phối giá nhà trên thị trường', fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel('Giá nhà (USD)', fontsize=11)
ax.set_ylabel('Số căn nhà', fontsize=11)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))

# Thêm đường trung bình
ax.axvline(df['Price'].mean(), color='#D85A30', linewidth=1.8, linestyle='--',
           label=f"Trung bình: ${df['Price'].mean()/1000:.0f}K")
ax.legend(fontsize=10)

ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('q1_price_distribution.png', dpi=150)
plt.show()

# In insight
print(f"Giá trung bình: ${df['Price'].mean():,.0f}")
print(f"Giá thấp nhất: ${df['Price'].min():,.0f}")
print(f"Giá cao nhất: ${df['Price'].max():,.0f}")
print(f"Độ lệch chuẩn: ${df['Price'].std():,.0f}")

Q2 — Diện tích có quyết định giá không?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('/content/House Price Prediction Dataset.csv')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Chart trái: Scatter plot Area vs Price ---
sample = df.sample(300, random_state=42)
axes[0].scatter(sample['Area'], sample['Price'],
                alpha=0.4, color='#378ADD', s=30, edgecolors='none')

# Trend line
z = np.polyfit(sample['Area'], sample['Price'], 1)
p = np.poly1d(z)
x_line = np.linspace(sample['Area'].min(), sample['Area'].max(), 100)
axes[0].plot(x_line, p(x_line), color='#D85A30', linewidth=1.5, linestyle='--')

corr = df['Area'].corr(df['Price'])
axes[0].set_title(f'Diện tích vs Giá  (r = {corr:.3f})', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Diện tích (ft²)', fontsize=10)
axes[0].set_ylabel('Giá ($)', fontsize=10)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))
axes[0].spines[['top', 'right']].set_visible(False)

# --- Chart phải: Correlation của tất cả biến số ---
features = ['Area', 'Bedrooms', 'Bathrooms', 'Floors', 'YearBuilt']
corrs = [df[f].corr(df['Price']) for f in features]
colors = ['#378ADD' if c > 0 else '#D85A30' for c in corrs]

bars = axes[1].barh(features, corrs, color=colors, edgecolor='none', height=0.5)
axes[1].axvline(0, color='gray', linewidth=0.8)
axes[1].set_title('Tương quan với giá nhà', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Hệ số tương quan (r)', fontsize=10)

for bar, val in zip(bars, corrs):
    axes[1].text(val + 0.002, bar.get_y() + bar.get_height()/2,
                 f'{val:.3f}', va='center', fontsize=9)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('Q2: Biến số nào tương quan nhất với giá nhà?', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('q2_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n→ Insight: Không có biến số nào có |r| > 0.1 — giá nhà gần như ngẫu nhiên với các đặc trưng vật lý!")

Q3 — Vị trí nào có giá cao nhất?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('/content/House Price Prediction Dataset.csv')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Chart trái: Bar chart giá trung bình theo Location ---
loc_stats = df.groupby('Location')['Price'].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False)

colors = ['#185FA5', '#378ADD', '#5a8cc4', '#85B7EB']
bars = axes[0].bar(loc_stats.index, loc_stats['mean'],
                   color=colors, edgecolor='none', width=0.55)

# Error bars (±1 std / sqrt(n))
axes[0].errorbar(range(len(loc_stats)), loc_stats['mean'],
                 yerr=loc_stats['std'] / np.sqrt(loc_stats['count']),
                 fmt='none', color='#444', capsize=4, linewidth=1.2)

axes[0].set_ylim(480000, 580000)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))
axes[0].set_title('Giá trung bình theo vị trí', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Giá trung bình ($)', fontsize=10)

for bar, val in zip(bars, loc_stats['mean']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1500,
                 f'${val/1000:.0f}K', ha='center', fontsize=10, fontweight='500')
axes[0].spines[['top', 'right']].set_visible(False)

# --- Chart phải: Box plot phân phối giá theo Location ---
order = loc_stats.index.tolist()
data_by_loc = [df[df['Location'] == loc]['Price'].values for loc in order]

bp = axes[1].boxplot(data_by_loc, labels=order, patch_artist=True,
                     medianprops=dict(color='white', linewidth=2),
                     flierprops=dict(marker='o', markerfacecolor='gray',
                                     markersize=3, alpha=0.4, linestyle='none'))

for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))
axes[1].set_title('Phân phối giá theo vị trí', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Giá ($)', fontsize=10)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('Q3: Vị trí ảnh hưởng như thế nào đến giá nhà?', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('q3_location.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nChênh lệch cao nhất - thấp nhất:")
print(f"  ${loc_stats['mean'].max() - loc_stats['mean'].min():,.0f} (~{(loc_stats['mean'].max()/loc_stats['mean'].min()-1)*100:.1f}%)")

Q4 — Tình trạng nhà có phản ánh đúng giá trị không?

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv('/content/House Price Prediction Dataset.csv')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Chart trái: Bar chart giá trung bình theo Condition ---
cond_order = ['Fair', 'Excellent', 'Good', 'Poor']
cond_means = df.groupby('Condition')['Price'].mean().reindex(cond_order)
cond_counts = df.groupby('Condition')['Price'].count().reindex(cond_order)

colors_cond = ['#3B6D11', '#639922', '#97C459', '#C0DD97']
bars = axes[0].bar(cond_order, cond_means.values, color=colors_cond,
                   edgecolor='none', width=0.55)

axes[0].set_ylim(490000, 580000)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))
axes[0].set_title('Giá trung bình theo tình trạng nhà', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Giá trung bình ($)', fontsize=10)

for bar, val, count in zip(bars, cond_means.values, cond_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1200,
                 f'${val/1000:.0f}K\n(n={count})', ha='center', fontsize=9.5)

# Highlight "Fair" là cao nhất — bất ngờ!
axes[0].annotate('Bất ngờ!\nFair đắt nhất?', xy=(0, cond_means['Fair']),
                 xytext=(1.2, 570000),
                 arrowprops=dict(arrowstyle='->', color='#D85A30', lw=1.5),
                 fontsize=9, color='#D85A30', fontweight='500')
axes[0].spines[['top', 'right']].set_visible(False)

# --- Chart phải: Garage effect + Condition kết hợp ---
combo = df.groupby(['Condition', 'Garage'])['Price'].mean().unstack()
combo = combo.reindex(cond_order)

x = np.arange(len(cond_order))
width = 0.35
axes[1].bar(x - width/2, combo['No'].values, width, label='Không có garage',
            color='#85B7EB', edgecolor='none')
axes[1].bar(x + width/2, combo['Yes'].values, width, label='Có garage',
            color='#185FA5', edgecolor='none')

axes[1].set_xticks(x)
axes[1].set_xticklabels(cond_order)
axes[1].set_ylim(490000, 590000)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${int(x/1000)}K'))
axes[1].set_title('Giá theo Tình trạng × Garage', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Giá trung bình ($)', fontsize=10)
axes[1].legend(fontsize=9)
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('Q4: Tình trạng nhà có quyết định giá trị thực không?', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('q4_condition.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n→ Insight: Garage ảnh hưởng không đáng kể — sự khác biệt chủ yếu là ngẫu nhiên trong dữ liệu này.")